### 1.2 Chunking: Dividindo Documentos em Pedaços

LLMs têm um limite de tokens no contexto. Além disso, documentos longeiros inteiros como contexto são ineficientes — a parte relevante é geralmente pequena.

**Chunking** é dividir documentos em pedaços menores. A estratégia mais simples: tamanho fixo com overlap.

Por que **overlap**? Se você cortar um documento em partes de 500 caracteres exatos, pode cortar uma frase no meio:
- Chunk 1: "...o algoritmo HNSW usa uma estrutura hierárquica de"
- Chunk 2: "grafos para permitir busca eficiente..."

Com overlap de 50 caracteres, o início do Chunk 2 repete o fim do Chunk 1, garantindo que a ideia completa apareça em pelo menos um chunk.

**Tamanhos típicos em produção:** 256-512 tokens para documentação técnica, 128-256 para FAQ, 512-1024 para artigos longos.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import numpy as np
import ollama
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# Stack completo
model = SentenceTransformer('all-MiniLM-L6-v2')
client = QdrantClient(host='localhost', port=6333, grpc_port=6334, prefer_grpc=True)

# Verificar Ollama
import httpx
try:
    r = httpx.get('http://localhost:11434/api/tags')
    modelos = [m['name'] for m in r.json().get('models', [])]
    LLM_MODEL = 'llama3.2' if any('llama3.2' in m for m in modelos) else (modelos[0] if modelos else None)
    print(f'LLM disponivel: {LLM_MODEL}')
except:
    LLM_MODEL = None
    print('Ollama nao disponivel — RAG sem geracao')

COLLECTION = 'naive_rag_docs'
print('Stack pronto!')

## Fase 1: INDEXING

### 1.1 Carregar Documentos

In [ ]:
# Carregar documentos de exemplo
docs_dir = Path('../data/sample_docs')
documentos = []

for filepath in docs_dir.glob('*.md'):
    texto = filepath.read_text(encoding='utf-8')
    documentos.append({
        'texto': texto,
        'fonte': filepath.name,
        'titulo': filepath.stem.replace('_', ' ').title(),
    })

print(f'Documentos carregados: {len(documentos)}')
for d in documentos:
    chars = len(d['texto'])
    print(f'  {d["titulo"]}: {chars:,} chars')

### 1.2 Chunking: Dividindo Documentos em Pedaços

LLMs têm um limite de tokens no contexto. Além disso, documentos longeiros inteiros como contexto são ineficientes — a parte relevante é geralmente pequena.

**Chunking** é dividir documentos em pedaços menores. A estratégia mais simples: tamanho fixo com overlap.

Por que **overlap**? Se você cortar um documento em partes de 500 caracteres exatos, pode cortar uma frase no meio:
- Chunk 1: "...o algoritmo HNSW usa uma estrutura hierárquica de"
- Chunk 2: "grafos para permitir busca eficiente..."

Com overlap de 50 caracteres, o início do Chunk 2 repete o fim do Chunk 1, garantindo que a ideia completa apareça em pelo menos um chunk.

**Tamanhos típicos em produção:** 256-512 tokens para documentação técnica, 128-256 para FAQ, 512-1024 para artigos longos.

In [ ]:
def chunk_texto(texto, chunk_size=800, overlap=100):
    """Chunking simples por caracter com overlap."""
    chunks = []
    start = 0
    while start < len(texto):
        end = min(start + chunk_size, len(texto))
        
        # Tentar terminar no fim de uma frase
        if end < len(texto):
            for sep in ['. ', '.\n', '\n\n', '\n']:
                pos = texto.rfind(sep, start, end)
                if pos != -1 and pos > start + chunk_size // 2:
                    end = pos + len(sep)
                    break
        
        chunk = texto[start:end].strip()
        if len(chunk) > 50:  # ignorar chunks muito curtos
            chunks.append(chunk)
        
        start = max(start + 1, end - overlap)
    return chunks

# Chunkar todos os documentos
todos_chunks = []
for doc in documentos:
    chunks = chunk_texto(doc['texto'])
    for i, chunk in enumerate(chunks):
        todos_chunks.append({
            'texto': chunk,
            'fonte': doc['fonte'],
            'titulo': doc['titulo'],
            'chunk_idx': i,
        })

print(f'Total de chunks: {len(todos_chunks)}')
print(f'Tamanho medio: {sum(len(c["texto"]) for c in todos_chunks)/len(todos_chunks):.0f} chars')
print(f'\nPrimeiro chunk:')
print(todos_chunks[0]['texto'][:300], '...')

### 1.3 Embeddings

In [ ]:
import time

textos_chunks = [c['texto'] for c in todos_chunks]

t0 = time.time()
embeddings = model.encode(
    textos_chunks,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=32,
)
elapsed = time.time() - t0

print(f'\nEmbeddings criados!')
print(f'Shape: {embeddings.shape}')
print(f'Tempo: {elapsed:.2f}s ({len(textos_chunks)/elapsed:.0f} chunks/s)')
print(f'Memoria: {embeddings.nbytes / 1024 / 1024:.2f} MB')

### 1.4 Indexar no Qdrant

In [ ]:
if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)
client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

points = [
    PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload=todos_chunks[i],
    )
    for i in range(len(todos_chunks))
]

# Inserir em batches
BATCH_SIZE = 50
for i in range(0, len(points), BATCH_SIZE):
    client.upsert(COLLECTION, points=points[i:i+BATCH_SIZE])

info = client.get_collection(COLLECTION)
print(f'Indexados {info.points_count} chunks na collection "{COLLECTION}"')
print('INDEXING COMPLETO!')

## Fase 2: QUERYING (Online, Tempo Real)

O usuário fez uma pergunta. Agora precisamos: encontrar os documentos mais relevantes e gerar uma resposta.

### 2.1 Retrieval

O retrieval tem dois passos:
1. **Embed a query**: a pergunta do usuário vira um vetor (usando o MESMO modelo da indexação)
2. **Busca vetorial**: encontrar os K chunks com vetores mais similares

**Por que usar o mesmo modelo?** Se você indexou com MiniLM-L6-v2, precisa fazer a query com MiniLM-L6-v2. Modelos diferentes geram espaços vetoriais diferentes — a similaridade não seria comparável.

**Latência típica:** embedding da query < 10ms, busca vetorial no Qdrant < 5ms para 100K vetores.

In [ ]:
def recuperar(pergunta, top_k=5):
    """Recupera os k chunks mais relevantes para a pergunta."""
    query_vec = model.encode(pergunta, normalize_embeddings=True)
    
    results = client.query_points(
        collection_name=COLLECTION,
        query=query_vec.tolist(),
        limit=top_k,
        with_payload=True,
    ).points
    
    return [
        {
            'texto': r.payload['texto'],
            'fonte': r.payload['fonte'],
            'score': r.score,
        }
        for r in results
    ]

# Teste de recuperacao
pergunta_teste = 'Como funciona o mecanismo de self-attention nos Transformers?'
chunks_recuperados = recuperar(pergunta_teste, top_k=3)

print(f'Pergunta: {pergunta_teste}')
print(f'\nChunks recuperados:')
for i, chunk in enumerate(chunks_recuperados):
    print(f'\n[{i+1}] Score: {chunk["score"]:.3f} | Fonte: {chunk["fonte"]}')
    print(chunk['texto'][:200], '...')

In [ ]:
RAG_PROMPT = """Voce e um assistente tecnico. Responda a pergunta baseado APENAS no contexto abaixo.
Se a resposta nao estiver no contexto, diga "Nao encontrei informacao suficiente sobre isso no contexto."

Contexto:
{contexto}

Pergunta: {pergunta}

Resposta:"""

def gerar_resposta(pergunta, chunks):
    """Gera resposta usando Ollama com contexto recuperado."""
    contexto = '\n\n---\n\n'.join(
        f'[{c["fonte"]}]\n{c["texto"]}' for c in chunks
    )
    
    prompt = RAG_PROMPT.format(contexto=contexto, pergunta=pergunta)
    
    response = ollama.chat(
        model=LLM_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return response['message']['content']

def rag(pergunta, top_k=5, verbose=True):
    """Pipeline RAG completo."""
    # 1. Recuperar
    chunks = recuperar(pergunta, top_k=top_k)
    
    # 2. Gerar
    if LLM_MODEL:
        resposta = gerar_resposta(pergunta, chunks)
    else:
        resposta = f'[LLM indisponivel] Contexto recuperado:\n' + '\n'.join(c['texto'][:100] for c in chunks)
    
    if verbose:
        print(f'Pergunta: {pergunta}')
        print(f'\nContexto ({len(chunks)} chunks recuperados):')
        for c in chunks:
            print(f'  - [{c["score"]:.3f}] {c["texto"][:80]}...')
        print(f'\nResposta RAG:\n{resposta}')
    
    return {'resposta': resposta, 'fontes': chunks}

# Testar o RAG
resultado = rag('O que e HNSW e como ele funciona?')

### Análise do pipeline completo

Agora você tem um sistema RAG funcional. Vamos analisar o que funciona bem e o que não funciona.

**Funciona bem:**
- Perguntas diretas sobre informações presentes nos documentos
- Documentos bem estruturados com parágrafos coesos
- Queries similares linguisticamente ao texto dos documentos

**Falha nesses casos:**
- Queries ambíguas ("como funciona?") sem contexto suficiente
- Perguntas que requerem combinar informação de múltiplos documentos distantes
- Perguntas com negação ("o que NÃO é...") — embeddings são ruins para negação
- Informação que está "nas entrelinhas", não explícita no texto

Esses problemas motivam as técnicas de Advanced RAG que veremos adiante: reescrita de query, re-ranking, compressão de contexto.

In [ ]:
# Mais perguntas de teste
perguntas = [
    'Quais sao os tipos de quantizacao vetorial?',
    'Como funciona backpropagation?',
    'Qual e a diferenca entre float32 e int8?',
]

for p in perguntas:
    print('\n' + '='*60)
    resultado = rag(p, verbose=True)

### 1.2 Chunking: Dividindo Documentos em Pedaços

LLMs têm um limite de tokens no contexto. Além disso, documentos longeiros inteiros como contexto são ineficientes — a parte relevante é geralmente pequena.

**Chunking** é dividir documentos em pedaços menores. A estratégia mais simples: tamanho fixo com overlap.

Por que **overlap**? Se você cortar um documento em partes de 500 caracteres exatos, pode cortar uma frase no meio:
- Chunk 1: "...o algoritmo HNSW usa uma estrutura hierárquica de"
- Chunk 2: "grafos para permitir busca eficiente..."

Com overlap de 50 caracteres, o início do Chunk 2 repete o fim do Chunk 1, garantindo que a ideia completa apareça em pelo menos um chunk.

**Tamanhos típicos em produção:** 256-512 tokens para documentação técnica, 128-256 para FAQ, 512-1024 para artigos longos.

## Resumo do Naive RAG

**O que construímos:** pipeline completo de RAG em ~100 linhas sem framework.

**Latências típicas:**
| Fase | Tempo | Nota |
|------|-------|------|
| Indexação (100K docs) | ~10 min | Uma vez só, offline |
| Embedding da query | ~10ms | Por request |
| Busca vetorial | ~5ms | Por request |
| Geração LLM | 2-8s | Domina o tempo total |

**Quando Naive RAG é suficiente:**
- Documentação técnica bem estruturada
- Base < 100K documentos
- Queries relativamente simples e diretas
- Protótipo ou MVP

**Quando você precisa de mais:**
- Qualidade de resposta insatisfatória após testes com usuários reais
- Queries complexas com múltiplos passos de raciocínio
- Documentos muito longos ou mal estruturados
- Necessidade de citar fontes com precisão

**Próximos passos:**
- [02 — Chunking Strategies](02_chunking_strategies.html): estratégias melhores de divisão de documentos
- [03 — Retrieval Strategies](03_retrieval_strategies.html): hybrid search, re-ranking, MMR